In [2]:
import json
import numpy as np
import requests
from rich import print as pprint

In [3]:
class CoinMarketCapApi:
    def __init__(self, headers: dict[str, str], base_url: str):
        self.headers = headers
        self.base_url = base_url

    def get(self, endpoint: str):
        response = requests.get(f"{self.base_url}{endpoint}", headers=self.headers)
        response.raise_for_status()
        return response.text

In [ ]:
headers = {
    'Accepts': 'application/json',
    'X-CMC_PRO_API_KEY': 'e889b068cbe843c489a0e30eb6804e8c'
}

base_url = 'https://pro-api.coinmarketcap.com'

api = CoinMarketCapApi(headers=headers, base_url=base_url)

In [ ]:
import time
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import requests


class CoinMarketCapScraper:
    def __init__(self, url: str) -> None:
        self.options = webdriver.ChromeOptions()
        # self.options.add_argument("--headless")  # No UI
        self.options.add_argument("--no-sandbox")
        self.options.add_argument("--window-size=1920,1080")  # Full screen żeby było widać wszystie elementy
        self.driver = webdriver.Chrome(options=self.options)
        self.driver.get(url)

    def get_crypto_price(self, delay: int = 5) -> str:
        time.sleep(delay)  # waiting between js refreshes the price
        price_element = self.driver.find_element(By.CSS_SELECTOR, '[data-test="text-cdp-price-display"]')
        return price_element.text

    def get_last_posts(self, delay: int = 10):
        # refresh site to get new posts
        self.driver.refresh()
        wait = WebDriverWait(self.driver, delay)
        
        # get 'latest' button and click it
        element = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '[data-index="tab-Latest"]')))
        element.click()
        # to make sure posts are loaded
        time.sleep(5)

        posts_div = self.driver.find_element(By.CSS_SELECTOR, '[data-test="coin-community-post-list"]')

        return posts_div

    def get_additional_info(self, delay: int = 5):
        time.sleep(delay)

In [ ]:
def save_posts(url: str, filepath: str, name: str):
    scraper = CoinMarketCapScraper(url)
    seen_posts = set()

    while True:
        day = datetime.now().strftime("%Y-%m-%d")
        with open(f"data/{filepath}_posts_{day}.txt", "a+", encoding="utf-8") as file:
            posts_div = scraper.get_last_posts(delay=10)

            # this is list of divs containing one whole post
            posts = posts_div.find_elements(By.CSS_SELECTOR, '[data-test="community-post"]')

            print(f"Found {len(posts)} posts for {name}.")
            for post in posts:
                post_id = post.get_attribute("data-post-id")
                if post_id in seen_posts:
                    continue

                seen_posts.add(post_id)

                text = post.find_element(By.CLASS_NAME, "text")
                full_text = text.get_attribute("textContent")
                if full_text:
                    file.write(f"{post_id};{full_text}\n\n")

In [38]:
url = 'https://coinmarketcap.com/currencies/bitcoin/'
btc_scraper = CoinMarketCapScraper(url)

### testing

In [15]:
posts_div = btc_scraper.get_last_posts()

In [17]:
posts = posts_div.find_elements(By.CSS_SELECTOR, '[data-test="community-post"]')

In [20]:
text = posts[0].find_element(By.CLASS_NAME, "text")
text.get_attribute("textContent")

"$BTC Bitcoin doesn't guess value. It enforces it. Read that again. Then tell me I'm wrong.#Bitcoin  #BTC"

In [39]:
save_posts(url, 'test_posts.txt', 'Bitcoin')

Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
Found 6 posts for Bitcoin.
F

KeyboardInterrupt: 

### Posts

In [5]:
base = '/v1/content'

In [10]:
f"{api.base_url}{base}/posts/latest"

'https://pro-api.coinmarketcap.com/v1/content/posts/latest'

In [11]:
api.get(f"{base}/posts/latest")

HTTPError: 403 Client Error: Forbidden for url: https://pro-api.coinmarketcap.com/v1/content/posts/latest

In [ ]:
from concurrent.futures import wait
import time
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

class CoinMarketCapScraper:
    def __init__(self, url: str) -> None:
        self.options = webdriver.ChromeOptions()
        self.options.add_argument("--headless")  # No UI
        self.options.add_argument("--no-sandbox")
        self.options.add_argument("--window-size=1920,1080") # Full screen żeby było widać wszystie elementy
        self.driver = webdriver.Chrome(options=self.options)
        self.driver.get(url)

    def get_crypto_price(self, delay: int = 5) -> str:
        time.sleep(delay)  # waiting between js refreshes the price
        price_element = self.driver.find_element(By.CSS_SELECTOR, '[data-test="text-cdp-price-display"]')
        return price_element.text

    def get_last_posts(self, delay: int = 5):
        wait = WebDriverWait(self.driver, delay)
        element = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, '[data-index="tab-Latest"]'))
        )
        element.click()
        posts_div = self.driver.find_element(By.CSS_SELECTOR, '[data-test="coin-community-post-list"]')
        
        return posts_div


In [ ]:
url = 'https://coinmarketcap.com/currencies/bitcoin/' 
scraper = CoinMarketCapScraper(url)
seen_posts = set()
div = scraper.get_last_posts()

In [ ]:
posts = div.find_elements(By.CSS_SELECTOR, '[data-test="community-post"]')
with open('posts.txt', 'w+') as f:
    for post in posts:
        post_id = post.get_attribute('data-post-id')
        if post_id in seen_posts:
            continue
        seen_posts.add(post_id)

        text = post.find_element(By.CLASS_NAME, 'text')
        full_text = text.get_attribute('textContent')
        print(full_text)
        print("\n\n")
        f.write(full_text + "\n")

In [3]:
site = requests.get('https://coinmarketcap.com/currencies/bitcoin/')

In [6]:
with open('btc.html', 'w') as f:
    f.write(site.text)